# TP 4 — Partie 2 : transporter le flux avec Kafka

Déposer des fichiers dans un dossier ne suffit pas en entreprise : plusieurs applications doivent lire les mêmes événements, chacune à son rythme, sans rien perdre si l'une d'elles s'arrête. C'est le rôle d'**Apache Kafka**.

| Notion | Définition | Dans ce TP |
|---|---|---|
| **broker** | le serveur Kafka, qui stocke les messages | le conteneur `kafka` (un seul broker) |
| **topic** | une file de messages, identifiée par un nom | `suivi_colis` |
| **partition** | un morceau du topic ; l'ordre des messages est garanti **dans** une partition | 3 partitions |
| **offset** | le numéro d'un message dans sa partition : 0, 1, 2… | position de lecture |
| **clé** | choisit la partition d'un message : même clé, même partition | `colis_id` |
| **producteur** | l'application qui envoie les messages | `rejouer_suivi.py` |
| **consommateur** et **groupe** | l'application qui lit ; un groupe mémorise où il en est | vos lecteurs Python, puis Spark |

Un message lu n'est **pas** supprimé : il reste dans le topic pendant la durée de rétention (7 jours par défaut). Chaque lecteur peut donc relire le flux depuis le début.

> **⚠️ Profil `streaming`.** Cette partie demande Kafka et son interface web. Démarrez-les depuis un terminal de votre machine (pas dans JupyterLab) :
> ```
> docker compose --profile streaming up -d kafka kafka-ui
> ```
> Kafka UI est alors disponible sur **http://localhost:8090**.

In [ ]:
import json
import time

from confluent_kafka import Consumer, TopicPartition
from confluent_kafka.admin import AdminClient, NewTopic

from flux import KAFKA, TOPIC, a_completer, rejouer, supprimer_groupes, supprimer_topic

admin = AdminClient({"bootstrap.servers": KAFKA})
supprimer_topic(TOPIC)                                   # on repart d'un topic absent
supprimer_groupes("tp4_lecteurs", "tp4_tableau_de_bord")
print("Kafka répond. Topics existants :", sorted(t for t in admin.list_topics(timeout=10).topics
                                                if not t.startswith("__")))

## Étape 1 — Créer le topic

Le broker du cours refuse de créer un topic automatiquement : on le crée explicitement, avec le nombre de partitions voulu.

Avec `admin.create_topics`, créez le topic `TOPIC` avec **3 partitions** et un **facteur de réplication de 1**. `create_topics` prend une liste de `NewTopic(nom, num_partitions=..., replication_factor=...)` et renvoie un dictionnaire de *futures* : appelez `.result()` sur chacun pour attendre la création.

In [ ]:
a_completer("création du topic")  # À COMPLÉTER
description = admin.list_topics(timeout=10).topics[TOPIC]
print(f"topic {TOPIC} : {len(description.partitions)} partitions")

> **❓ Question 1.** Que signifie un facteur de réplication de 1 ? Pourquoi est-ce acceptable ici et pas en production ?

## Étape 2 — Envoyer une semaine d'événements

On rejoue la première semaine de mars 2018, **sans pause** : tout est envoyé d'un coup.

In [ ]:
n_envoyes = rejouer("--depuis", "2018-03-01", "--jusqu-a", "2018-03-08", "--sans-pause")

Ouvrez **Kafka UI** (http://localhost:8090), puis *Topics > suivi_colis* :
- l'onglet **Overview** montre le nombre de messages de chaque partition ;
- l'onglet **Messages** montre les messages eux-mêmes : clé, valeur, partition, offset.

Comparez avec le décompte suivant.

In [ ]:
lecteur = Consumer({"bootstrap.servers": KAFKA, "group.id": "tp4_verification"})
total = 0
for p in range(3):
    debut, fin = lecteur.get_watermark_offsets(TopicPartition(TOPIC, p), timeout=10)
    print(f"partition {p} : offsets {debut} à {fin - 1}, soit {fin - debut} messages")
    total += fin - debut
lecteur.close()
print("total :", total, "| envoyés :", n_envoyes)

## Étape 3 — Lire les messages

Un **consommateur** s'abonne au topic et appelle `poll()` en boucle : chaque appel renvoie un message, ou `None` si rien n'est arrivé dans le délai.

Complétez la boucle : lisez **10 messages** avec le consommateur `lecteur` (groupe `tp4_lecteurs`) et affichez pour chacun sa partition, son offset, sa clé et le début de sa valeur. Un message `msg` offre `msg.partition()`, `msg.offset()`, `msg.key()` et `msg.value()` (en octets : `.decode()`).

In [ ]:
lecteur = Consumer({"bootstrap.servers": KAFKA, "group.id": "tp4_lecteurs",
                    "auto.offset.reset": "earliest",    # sans position mémorisée : depuis le début
                    "enable.auto.commit": False})        # on valide la position nous-mêmes
lecteur.subscribe([TOPIC])

lus = 0
while lus < 10:
    msg = lecteur.poll(5)
    if msg is None:
        continue
    # À COMPLÉTER : afficher partition, offset, clé et début de la valeur, puis compter le message
    a_completer("lecture d'un message")
lecteur.commit(asynchronous=False)     # on mémorise la position du groupe dans Kafka
lecteur.close()

> **❓ Question 2.** Les messages lus viennent-ils d'une seule partition ou de plusieurs ? Dans quel ordre ? Relancez la cellule : les mêmes messages reviennent-ils ? Pourquoi ?

## Étape 4 — La clé garde l'ordre d'un colis

Le producteur envoie chaque événement avec la clé `colis_id`. Kafka calcule la partition à partir de la clé : tous les événements d'un même colis vont dans la **même** partition, donc restent dans l'ordre.

Le code suivant relit tout le topic et vérifie cette propriété, puis affiche les événements d'un colis.

In [ ]:
lecteur = Consumer({"bootstrap.servers": KAFKA, "group.id": "tp4_verification",
                    "auto.offset.reset": "earliest", "enable.auto.commit": False})
lecteur.assign([TopicPartition(TOPIC, p, 0) for p in range(3)])   # lecture directe, sans groupe
partitions_du_colis, evenements, vides = {}, {}, 0
while vides < 3:
    msg = lecteur.poll(2)
    if msg is None:
        vides += 1
        continue
    cle = msg.key().decode() if msg.key() else None
    partitions_du_colis.setdefault(cle, set()).add(msg.partition())
    evenements.setdefault(cle, []).append((msg.partition(), msg.offset(), msg.value().decode()))
lecteur.close()

print("colis :", len(partitions_du_colis))
print("colis répartis sur plusieurs partitions :",
      sum(1 for k, p in partitions_du_colis.items() if k and len(p) > 1))
exemple = max((k for k in evenements if k), key=lambda k: len(evenements[k]))
print(f"\nÉvénements du colis {exemple} :")
for partition, offset, valeur in evenements[exemple]:
    e = json.loads(valeur) if valeur.endswith("}") else {}
    print(f"  partition {partition}, offset {offset:>4} : {e.get('horodatage')}  {e.get('code')}")

> **❓ Question 3.** Pourquoi est-il important que les événements d'un même colis restent dans l'ordre ? Que se passerait-il si le producteur envoyait les messages **sans clé** ?

## Étape 5 — Deux groupes, deux lectures indépendantes

Un deuxième lecteur, le groupe `tp4_tableau_de_bord`, lit maintenant tout le topic. Ensuite, on calcule le **retard** (*lag*) de chaque groupe : le nombre de messages qui lui restent à lire dans chaque partition.

In [ ]:
tableau = Consumer({"bootstrap.servers": KAFKA, "group.id": "tp4_tableau_de_bord",
                    "auto.offset.reset": "earliest", "enable.auto.commit": False})
tableau.subscribe([TOPIC])
lus, vides = 0, 0
while vides < 3:
    msg = tableau.poll(2)
    if msg is None:
        vides += 1
        continue
    lus += 1
tableau.commit(asynchronous=False)
tableau.close()
print("messages lus par tp4_tableau_de_bord :", lus)


def retard(groupe):
    c = Consumer({"bootstrap.servers": KAFKA, "group.id": groupe})
    positions = c.committed([TopicPartition(TOPIC, p) for p in range(3)], timeout=10)
    total = 0
    for tp in positions:
        fin = c.get_watermark_offsets(tp, timeout=10)[1]
        pos = tp.offset if tp.offset >= 0 else 0
        print(f"  {groupe:22} partition {tp.partition} : position {pos:>5}, reste {fin - pos:>5}")
        total += fin - pos
    c.close()
    return total


for groupe in ("tp4_lecteurs", "tp4_tableau_de_bord"):
    print(groupe, ": retard total", retard(groupe))

Dans Kafka UI, l'onglet **Consumers** montre les mêmes groupes et leur retard (*Consumer Lag*).

> **❓ Question 4.** Pourquoi les deux groupes n'ont-ils pas le même retard ? Si le groupe `tp4_lecteurs` s'arrêtait une journée, qu'arriverait-il aux messages qu'il n'a pas lus ? Quelle différence avec une file de messages classique, où un message lu disparaît ?

## À retenir

- Un **topic** est découpé en **partitions** ; l'ordre n'est garanti que dans une partition, et la **clé** décide de la partition.
- Les messages restent dans Kafka après lecture : chaque **groupe** mémorise sa position (son **offset**) et lit à son rythme.
- Le **retard** d'un groupe indique s'il suit le flux ou s'il prend du retard.

Dans la partie 3, Spark devient le lecteur : il écrit le flux dans la couche Bronze du Lakehouse.